# Data Cleaning — Chicago Traffic Crashes

Dataset: [Traffic Crashes - Crashes](https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if/data_preview), City of Chicago Data Portal.

Goal of the wider project: identify what factors make a traffic crash more or less likely to
result in personal injury. That goal drives several of the cleaning decisions below (e.g. which
columns would leak the target, which columns are too sparse or too high-cardinality to use).

This notebook only **explores and documents** the cleaning decisions. Once we agree on them, the
actual cleaning function should move into `src/data3800_gruppe_7/` (per the README convention:
reusable loading/cleaning/feature code lives in `src/`, not in notebooks) so every notebook can
`from data3800_gruppe_7 import clean` and get the same cleaned frame.

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## Load the raw dataset

`data/` is gitignored (it's 650MB+, see `data_fetch.py` for how to fetch it), so every teammate
needs to run this against their own local copy.

In [ ]:
DATA_PATH = "../data/raw_dataset.csv"

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(df_raw.shape)
df_raw.head()

## Step 1 — First look

Before touching anything, get a sense of size, dtypes, and obvious missingness.

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="number").T

## Step 2 — Missingness audit

The pandas dtype a column got on load (`str`/`object` vs `int64`/`float64`) already tells us
something: columns pandas was forced to read as `float64` (e.g. `INJURIES_TOTAL`,
`BEAT_OF_OCCURRENCE`) contain `NaN`s even though they're conceptually integer counts/codes.

Ranking columns by missing % surfaces three distinct groups that need different treatment:

In [ ]:
missing_pct = (df_raw.isna().mean() * 100).round(2).sort_values(ascending=False)
missing_pct[missing_pct > 0]

**Group A — "indicator" flag columns that are ~70-100% missing**
(`WORKERS_PRESENT_I`, `DOORING_I`, `WORK_ZONE_TYPE`, `WORK_ZONE_I`, `PHOTOS_TAKEN_I`,
`STATEMENTS_TAKEN_I`, `NOT_RIGHT_OF_WAY_I`, `CRASH_DATE_EST_I`, `INTERSECTION_RELATED_I`,
`HIT_AND_RUN_I`). These are Chicago's "checkbox" fields: the officer only marks `Y`/`N` when it's
relevant, otherwise the field is left blank — blank is not random missingness, it's an implicit
"not applicable / no". Two reasonable options:
- Columns that are ~95%+ empty (`WORKERS_PRESENT_I`, `DOORING_I`, `WORK_ZONE_TYPE`, `WORK_ZONE_I`,
  `PHOTOS_TAKEN_I`, `STATEMENTS_TAKEN_I`, `NOT_RIGHT_OF_WAY_I`, `CRASH_DATE_EST_I`): drop them —
  there's almost no signal left once 95%+ of rows share the same (missing) value.
- The two that are still meaningfully populated (`INTERSECTION_RELATED_I` ~23% filled,
  `HIT_AND_RUN_I` ~31% filled) and `LANE_CNT` (~18% filled): `fillna("N")` for the two flags
  (blank really does mean "not flagged"), and treat `LANE_CNT`'s blank as "unknown" rather than
  imputing a lane count.

**Group B — small, genuinely-missing fields (<4%)**
(`REPORT_TYPE`, `LOCATION`/`LATITUDE`/`LONGITUDE`, `IDOT_CONTROL_NO`, and the injury columns incl.
`MOST_SEVERE_INJURY` at 0.22%). Small enough to just drop the affected rows, or impute a
placeholder category (`"UNKNOWN"`) for categorical ones — dropping ~2,400 rows out of 1.09M to
keep a clean target column is cheap.

**Group C — not missing, but still needs scrutiny**
`LANE_CNT` has a `99` category — that's very likely a sentinel for "unknown", not a 99-lane road
— check for sentinel-style placeholder values generally before treating a column as clean.

## Step 3 — Exact duplicates

Crash records can legitimately repeat an address/time if two unrelated crashes happen at the same
spot, so check the unique crash ID (not just full-row duplicates).

In [ ]:
print("duplicate CRASH_RECORD_ID:", df_raw["CRASH_RECORD_ID"].duplicated().sum())
print("fully duplicate rows:       ", df_raw.duplicated().sum())

No duplicates on either check for this dataset — one row per crash, as expected. Still worth checking after any merge/join in later notebooks.

## Step 4 — Data types & redundant derived columns

`CRASH_DATE` and `DATE_POLICE_NOTIFIED` load as strings, not `datetime`, so date arithmetic
(seasonality, reporting lag) won't work until they're parsed. The dataset also ships three columns
(`CRASH_MONTH`, `CRASH_HOUR`, `CRASH_DAY_OF_WEEK`) that are just `CRASH_DATE` pre-extracted —
confirm they agree with the parsed date before relying on either representation, then keep
whichever is more convenient instead of carrying both.

In [ ]:
crash_date = pd.to_datetime(df_raw["CRASH_DATE"], format="%m/%d/%Y %I:%M:%S %p")
police_notified = pd.to_datetime(df_raw["DATE_POLICE_NOTIFIED"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

print("CRASH_MONTH matches parsed date:", (crash_date.dt.month == df_raw["CRASH_MONTH"]).all())
print("CRASH_HOUR matches parsed date: ", (crash_date.dt.hour == df_raw["CRASH_HOUR"]).all())
print("date range:", crash_date.min(), "->", crash_date.max())
crash_date.dt.year.value_counts().sort_index()

`CRASH_MONTH`/`CRASH_HOUR` agree with the parsed timestamp, so they're redundant but not wrong
— fine to drop once `CRASH_DATE` is parsed to datetime.

Row counts by year show 2013-2015 are a ramp-up period (2 rows in 2013, 9.8k in 2015, vs. ~100k+ in
every full year since 2016) — almost certainly reporting-system coverage, not a shrinking number of
real crashes. Rows before ~2016 should probably be excluded so time-based features aren't skewed by
an artificially tiny early sample.

## Step 5 — Validity / range checks

Columns that parsed as numeric aren't automatically *valid*. Three checks worth doing on this
dataset specifically:

In [ ]:
# 1. Posted speed limit — look for implausible values (data entry errors)
df_raw["POSTED_SPEED_LIMIT"].value_counts().sort_index()

In [ ]:
# 2. Lat/long — rows sitting at (0, 0) are a geocoding-failure sentinel, not a real location
bad_coords = (df_raw["LATITUDE"] == 0) | (df_raw["LONGITUDE"] == 0)
print("rows with (lat, long) == 0:", bad_coords.sum())

# 3. Reporting lag — police notified before the crash happened is impossible
lag_hours = (police_notified - crash_date).dt.total_seconds() / 3600
print("negative lag (impossible):", (lag_hours < 0).sum())
print("lag > 24h (plausible for desk reports, but worth flagging):", (lag_hours > 24).sum())

- `POSTED_SPEED_LIMIT` has a long tail of clearly-wrong values (`0`, `1`-`9` mph, up to `99`)
  sitting alongside the expected 25/30/35/40/45 clusters — these look like data entry errors or
  "unknown" sentinels rather than real speed limits. Cap/bucket to plausible road speeds (e.g.
  5-70mph) and treat the rest as missing.
- A handful of rows have `(0, 0)` coordinates — set those to `NaN` rather than treating Chicago's
  south pole/equator intersection as a real crash site.
- Reporting lag is essentially always ≥ 0 (one single-row exception), so that field is trustworthy;
  the >24h cases are desk reports, a known, legitimate `REPORT_TYPE`, not data errors.

## Step 6 — Pick the target, and watch for leakage

The project question is "what factors make a crash more likely to result in injury." Two columns
could serve as the target:

- `CRASH_TYPE` — binary (`NO INJURY / DRIVE AWAY` vs `INJURY AND / OR TOW DUE TO CRASH`).
- `MOST_SEVERE_INJURY` — ordinal, 5 levels (`NO INDICATION OF INJURY` → `FATAL`), which matches the
  *ordered probit* approach used in both literature references in the README.

Whichever is chosen, the `INJURIES_*` count columns (`INJURIES_TOTAL`, `INJURIES_FATAL`,
`INJURIES_INCAPACITATING`, `INJURIES_NON_INCAPACITATING`, `INJURIES_REPORTED_NOT_EVIDENT`,
`INJURIES_NO_INDICATION`, `INJURIES_UNKNOWN`) are computed *from* the same outcome and must be
excluded from the feature set — they'd leak the target almost perfectly (confirmed below:
`INJURIES_TOTAL` is exactly the sum of the severity sub-counts).

In [ ]:
parts = df_raw[["INJURIES_FATAL", "INJURIES_INCAPACITATING",
                "INJURIES_NON_INCAPACITATING", "INJURIES_REPORTED_NOT_EVIDENT"]]
mask = parts.notna().all(axis=1)
print("INJURIES_TOTAL == sum of severity parts for all non-null rows:",
      (parts[mask].sum(axis=1) == df_raw.loc[mask, "INJURIES_TOTAL"]).all())

df_raw["MOST_SEVERE_INJURY"].value_counts(dropna=False)

## Step 7 — High-cardinality identifier columns

Some columns are identifiers or free-text-like fields, not predictive features — they'd either do
nothing in a model (pure IDs) or blow up one-hot encoding with thousands of near-unique categories:

- `CRASH_RECORD_ID`, `IDOT_CONTROL_NO` — unique per crash, pure IDs → drop.
- `STREET_NAME` (1,677 unique), `LOCATION` (360,780 unique), `STREET_NO`, `STREET_DIRECTION`,
  `BEAT_OF_OCCURRENCE` — precise location is better captured via `LATITUDE`/`LONGITUDE` directly
  (or a coarser derived feature like police district/ward) than via the raw street text.
- `PRIM_CONTRIBUTORY_CAUSE` / `SEC_CONTRIBUTORY_CAUSE` (40 categories each) are fine to keep as
  categorical features, but check for an `"UNABLE TO DETERMINE"` style placeholder category and
  decide whether to keep it as "unknown cause" or drop those rows.

In [ ]:
for c in ["STREET_NAME", "LOCATION", "PRIM_CONTRIBUTORY_CAUSE", "SEC_CONTRIBUTORY_CAUSE"]:
    print(f"{c}: {df_raw[c].nunique()} unique values")

## Summary — proposed cleaning pipeline

1. **Drop near-empty flag columns** (≥95% missing): `WORKERS_PRESENT_I`, `DOORING_I`,
   `WORK_ZONE_TYPE`, `WORK_ZONE_I`, `PHOTOS_TAKEN_I`, `STATEMENTS_TAKEN_I`, `NOT_RIGHT_OF_WAY_I`,
   `CRASH_DATE_EST_I`.
2. **Fill implicit-"no" flags**: `INTERSECTION_RELATED_I`, `HIT_AND_RUN_I` → `fillna("N")`.
3. **Parse dates**: `CRASH_DATE`, `DATE_POLICE_NOTIFIED` → `datetime64`; drop the now-redundant
   `CRASH_MONTH`/`CRASH_HOUR`/`CRASH_DAY_OF_WEEK` (or keep for convenience, your call).
4. **Filter to reliable years**: drop rows before ~2016 (reporting ramp-up).
5. **Clean invalid values**: bucket/cap implausible `POSTED_SPEED_LIMIT`s; set `(0, 0)` coordinates
   and the `LANE_CNT == 99` sentinel to `NaN`.
6. **Drop rows missing the target**: `MOST_SEVERE_INJURY` (or `CRASH_TYPE`) is small enough
   (~0.2%) to drop rather than impute.
7. **Drop ID / high-cardinality / leakage columns**: `CRASH_RECORD_ID`, `IDOT_CONTROL_NO`,
   `STREET_NAME`, `LOCATION`, `STREET_NO`, `STREET_DIRECTION`, plus every `INJURIES_*` count column
   if the target is injury severity.
8. **Cast remaining categoricals to `category` dtype** to cut memory (the full frame is ~2.2GB as
   loaded) and speed up downstream encoding (`OneHotEncoder`/`OrdinalEncoder`, imported above).

Next step: once this list is agreed on, implement it as a single `clean_crashes(df)` function in
`src/data3800_gruppe_7/`, import it here with `%autoreload`, and keep this notebook as the
EDA/rationale trail rather than the pipeline itself.